In [1]:
import csv
import json
import random

In [2]:
age_ranges = {'Child (1-12)':range(1,13), 'Teen (13-18)':range(13,19), 'Young Adult (19-30)':range(19,31),
        'Adult (31-50)':range(31,51), 'Senior (51-70)':range(51,71), 'Elder (71-100)':range(71,101)}
age_groups = list(age_ranges.keys())
relationship_types = ['Family', 'Friendly', 'Romantic', 'Education', 'Household', 'Work', 'Social']

In [3]:
def get_age_group(age):
    if age < 13:
        return age_groups[0]
    if age < 19:
        return age_groups[1]
    if age < 31:
        return age_groups[2]
    if age < 51:
        return age_groups[3]
    return age_groups[4]

In [4]:
genders = ["male", "female", "other"]

# **Hobbies**

In [ ]:
def load_hobbies(path='hobbies_by_category.csv'):
    hobbies = []
    hobby_categories = []
    hobbies_by_category = {}
    hobbies_to_category = {}
    hobbies_by_age_category = {}

    hobby_ages = {}
    with open(path, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)

        for row in reader:
            if row:
                hobbies.append(row[0])
                if hobby_categories == [] or hobby_categories[-1] != row[1]:
                    hobby_categories.append(row[1])
                    hobbies_by_category[row[1]] = []
                hobbies_to_category[row[0]] = row[1]
                ages = row[2].split(', ')
                hobby_ages[row[0]] = ages
                hobbies_by_category[row[1]].append(row[0])
                for a in ages:
                    if (a,row[1]) not in hobbies_by_age_category.keys():
                        hobbies_by_age_category[(a,row[1])] = [row[0]]
                    else:
                        hobbies_by_age_category[(a,row[1])].append(row[0])

    return hobbies, hobby_categories, hobbies_by_category, hobbies_to_category, hobbies_by_age_category, hobby_ages

hobbies, hobby_categories, hobbies_by_category, hobbies_to_category, hobbies_by_age_category, hobby_ages = load_hobbies()

# **Relationships**

In [ ]:
def load_relationships(path='relationships.csv'):
    relationships = []
    relationship_types = []

    relationships_by_age = {age:[] for age in age_groups}
    relationships_by_gender = {gender:[] for gender in genders}
    relationships_by_type = {}
    relationship_to_type = {}
    relationships_by_age_gender = {(age, gender):[] for age in age_groups for gender in genders}
    relationships_by_age_gender_type = {}

    with open(path, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)

        for row in reader:
            if row:
                relationships.append(row[0])
                if relationship_types == [] or relationship_types[-1] != row[1]:
                    relationship_types.append(row[1])

                ags = row[2].split(', ')
                gnds = row[3].split(', ')
                relationship_to_type[row[0]] = row[1]
                for a in ags:
                    relationships_by_age[a].append(row[0])
                    for g in gnds:
                        relationships_by_age_gender[(a, g)].append(row[0])
                        if (a, g, row[1]) not in relationships_by_age_gender_type.keys():
                            relationships_by_age_gender_type[(a, g, row[1])] = [row[0]]
                        else:
                            relationships_by_age_gender_type[(a, g, row[1])].append(row[0])
                for g in gnds:
                    relationships_by_gender[g].append(row[0])

                if row[1] not in relationships_by_type.keys():
                    relationships_by_type[row[1]] = [row[0]]
                else:
                    relationships_by_type[row[1]].append(row[0])

    return relationships, relationship_types, relationships_by_age, relationships_by_gender, \
            relationships_by_type, relationship_to_type, relationships_by_age_gender, \
            relationships_by_age_gender_type

relationships, relationship_types, relationships_by_age, relationships_by_gender, \
            relationships_by_type, relationship_to_type, relationships_by_age_gender, \
            relationships_by_age_gender_type = load_relationships()

# **Occasions**

In [ ]:
def load_occasions(path='occasions.csv'):
    occasions = []
    occassions_by_age_gender_relationshiptype = {(age, gender, relationship): ['other']
           for age in age_groups
           for gender in genders
           for relationship in relationship_types}
    occassions_to_age_gender_relationshiptype = {}

    with open(path, newline='', encoding='utf-8') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)

        for row in reader:
            if row:
                occasions.append(row[0])
                ags = row[2].split(', ')
                if row[1] != 'male' and row[1] != 'memale':
                    gndrs = genders
                else:
                    gndrs = [row[1]]
                rlnshps = row[3].split(', ')

                occassions_to_age_gender_relationshiptype[row[0]] = (ags, gndrs, rlnshps)
                for a in ags:
                    for g in gndrs:
                        for r in rlnshps:
                            occassions_by_age_gender_relationshiptype[(a, g, r)].append(row[0])
    return occasions, occassions_by_age_gender_relationshiptype, occassions_to_age_gender_relationshiptype

occasions, occassions_by_age_gender_relationshiptype, occassions_to_age_gender_relationshiptype = load_occasions()

# **Users**

In [ ]:
# Generate diverse user profiles
def generate_user_profiles(n=300):
    profiles = []
    for _ in range(n):
        age = random.randint(1, 100)
        gender = random.choice(genders)
        age_group = get_age_group(age)
        hobbs = random.sample(hobbies, k=random.randint(1, 8))

        user = {
            "age": age,
            "gender": gender,
            "hobbies": hobbs,
        }

        relationship = random.choice(relationships_by_age_gender[(age_group, gender)])
        relationship_type = relationship_to_type[relationship]
        occasion = random.choice(occassions_by_age_gender_relationshiptype[(age_group, gender, relationship_type)])

        profiles.append({
            "user": user,
            "relationship": relationship,
            "occasion": occasion
        })
    return profiles

In [ ]:
# Load users from JSON
def load_users(file_path="user_profiles.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        if isinstance(data, dict):
            return list(data.values())
        return data

users = load_users()

# **Products**

In [6]:
def load_products(file_path="amazon_products_random_10k_with_categories.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

        products = []
        for product in data:
            products.append({
                "asin": product.get("asin"),
                "title": product.get("title"),
                "category": product.get("category"),
            })
        return products

products = load_products()

In [7]:
products_by_category = {}
for product in products:
    if product['category'] in products_by_category.keys():
        products_by_category[product['category']].append(product)
    else:
        products_by_category[product['category']] = [product]

In [ ]:
products_by_relationship = {relationship: [] for relationship in relationships}
bad_products_by_relationship = {relationship: [] for relationship in relationships}

for relationship in relationships:
    for product in products:
        ptitle = product['title'].lower()
        if relationship.lower() in ptitle:
            products_by_relationship[relationship].append(product)
        else:
            for r in relationships:
                if r!=relationship and r.lower() in ptitle:
                    bad_products_by_relationship[relationship].append(product)
                    break

In [ ]:
products_by_occasion = {occasion: [] for occasion in occasions}
bad_products_by_occasion = {occasion: [] for occasion in occasions}

for occasion in occasions:
    for product in products:
        ptitle = product['title'].lower()
        if occasion.lower() in ptitle:
            products_by_occasion[occasion].append(product)
        else:
            for o in occasions:
                if o!=occasion and o.lower() in ptitle:
                    bad_products_by_occasion[occasion].append(product)
                    break


In [ ]:
products_by_hobbies = {hobby: [] for hobby in hobbies}
products_by_hobbies_and_categories = {hobby: [] for hobby in hobbies}
for hobby in hobbies:
    for product in products:
        ptitle = product['title'].lower()
        if hobby.lower() in ptitle:
            products_by_hobbies[hobby].append(product)
            pcat = product['category'].lower()
            if hobby.lower() in pcat:
                products_by_hobbies_and_categories[hobby].append(product)


In [ ]:
products_by_age_and_gender = {(age_group, gender):[] for age_group in age_groups
                              for gender in genders}
bad_products_by_age_and_gender = {(age_group, gender):[] for age_group in age_groups
                              for gender in genders}
for product in products:
    ptitle = product['title'].lower()
    if 'boy' in ptitle:
        products_by_age_and_gender[('Child (1-12)', 'male')].append(product)
        for age_group in age_groups[1:]:
            for gender in genders:
                bad_products_by_age_and_gender[(age_group, gender)].append(product)
    if 'girl' in ptitle:
        products_by_age_and_gender[('Child (1-12)', 'female')].append(product)
        for age_group in age_groups[1:]:
            for gender in genders:
                bad_products_by_age_and_gender[(age_group, gender)].append(product)
    if 'kid' in ptitle or 'toy' in ptitle or 'baby' in ptitle or 'toddler':
        for gender in genders:
            products_by_age_and_gender[('Child (1-12)', gender)].append(product)
        for age_group in age_groups[1:]:
            for gender in genders:
                bad_products_by_age_and_gender[(age_group, gender)].append(product)
    if 'teen' in ptitle:
        for gender in genders:
            products_by_age_and_gender[('Teen (13-18)', gender)].append(product)
    if ' men' in ptitle and not 'women' in ptitle:
        for age_group in age_groups[2:]:
            products_by_age_and_gender[(age_group, 'male')].append(product)
        for age_group in age_groups:
            bad_products_by_age_and_gender[(age_group, 'female')].append(product)
        for gender in genders:
            bad_products_by_age_and_gender[('Child (1-12)', gender)].append(product)
    if 'women' in ptitle:
        for age_group in age_groups[2:]:
            products_by_age_and_gender[(age_group, 'female')].append(product)
        for age_group in age_groups:
            bad_products_by_age_and_gender[(age_group, 'male')].append(product)
        for gender in genders:
            bad_products_by_age_and_gender[('Child (1-12)', gender)].append(product)

# **Dataset Synthesis**

In [ ]:
def sample(population, k):
    return random.sample(population, min(k, len(population)))

In [ ]:
print('\033[1mPossible Age Groups:\033[0m')
for g in age_groups:
    print('-\t', g)
print('\033[1mPossible Genders\033[0m:')
for g in genders:
    print('-\t', g)
print('\033[1mPossible Hobby Categories\033[0m:')
for c in hobby_categories:
    print('-\t', c)
print('\033[1mPossible Product Categories\033[0m:')
for c in products_by_category:
    print('-\t', c)
print('\033[1mPossible Relationship Types\033[0m:')
for t in relationship_types:
    print('-\t', t)

Possible Age Groups:
-	 Child (1-12)
-	 Teen (13-18)
-	 Young Adult (19-30)
-	 Adult (31-50)
-	 Senior (51-70)
-	 Elder (71-100)
Possible Genders:
-	 male
-	 female
-	 other
Possible Hobby Categories:
-	 Creative & Artistic
-	 Music & Performing Arts
-	 Ball & Team Sports
-	 Gaming & Puzzles
-	 Intellectual & Educational
-	 DIY & Technical
-	 Physical & Fitness
-	 Outdoor & Nature
-	 Animals & Pets
-	 Content & Media
-	 Cleaning & Organizing
-	 Entertainment & Leisure
-	 Social & Lifestyle
-	 Wellness & Spirituality
-	 Culinary & Beverage
-	 Financial & Investment
-	 Collecting & Curating
-	 Travel & Exploration
-	 Automotive & Motorsports
Possible Product Categories:
-	 PlayStation 5 Consoles, Games & Accessories
-	 Smart Home: WiFi and Networking
-	 Security & Surveillance Equipment
-	 Beauty Tools & Accessories
-	 Computer Networking
-	 Men's Shoes
-	 Computer Components
-	 Finger Toys
-	 Lights, Bulbs & Indicators
-	 Automotive Performance Parts & Accessories
-	 Vehicle Electronics

## **Hobbies**

In [ ]:
hobbies_dataset = []

In [ ]:
def generate_hobby_category_specific_entries(_hobby_category, n=100,
                                      _product_categories=list(products_by_category.keys()), score=1):
    possible_products = []
    for cat in _product_categories:
        possible_products.extend(products_by_category[cat])

    entries = []
    for i in range(n):
        _hobby = random.choice(hobbies_by_category[_hobby_category])

        _product = random.choice(possible_products)

        entries.append({
            'hobbies': _hobby,
            'product': _product,
            'suitability': score
        })

    return entries

In [ ]:
def generate_hobby_specific_entries(n=1000):
    entries = []
    for i in range(n):
        hobby = random.choice(hobbies)

        good_prods = products_by_hobbies[hobby]
        if good_prods != []:
            prods = sample(good_prods, 5)
            for product in prods:
                entries.append({
                    'hobbies': hobby,
                    'product': product,
                    'suitability': 0.95
                })

        best_prods = products_by_hobbies_and_categories[hobby]
        if best_prods != []:
            prods = sample(best_prods, 5)
            for product in prods:
                entries.append({
                    'hobbies': hobby,
                    'product': product,
                    'suitability': 1
                })

    return entries

entries = generate_hobby_specific_entries()
hobbies_dataset.extend(entries)


### Generate positive samples by hobby category

In [ ]:
positive_combos = [
    {
        'hobby_category': 'Creative & Artistic',
        'product_categories': [
            'Arts & Crafts Supplies', 'Painting, Drawing & Art Supplies', 'Fabric Decorating',
            'Craft Supplies & Materials', 'Craft & Hobby Fabric', 'Printmaking Supplies',
            'Beading & Jewelry Making', 'Scrapbooking & Stamping Supplies', 'Needlework Supplies',
            'Knitting & Crochet Supplies', 'Arts, Crafts & Sewing Storage','Sewing Products'
        ]
    },
    {
        'hobby_category': 'Music & Performing Arts',
        'product_categories': [
            'Headphones & Earbuds', 'Portable Audio & Video', 'Home Audio & Theater Products',
            'Smart Home: Home Entertainment', 'Camera & Photo', 'Video Projectors'
        ]
    },
    {
        'hobby_category': 'Ball & Team Sports',
        'product_categories': [
            'Sports & Outdoors', 'Sports & Fitness', 'Sports Nutrition Products',
            'Sports & Outdoor Play Toys', 'Games & Accessories', 'Diet & Sports Nutrition',
            'Outdoor Recreation'
        ]
    },
    {
        'hobby_category': 'Gaming & Puzzles',
        'product_categories': [
            'PlayStation 5 Consoles, Games & Accessories', 'PlayStation 4 Games, Consoles & Accessories',
            'PlayStation 3 Games, Consoles & Accessories', 'PlayStation Vita Games, Consoles & Accessories',
            'Xbox Series X & S Consoles, Games & Accessories', 'Xbox One Games, Consoles & Accessories',
            'Xbox 360 Games, Consoles & Accessories', 'Nintendo Switch Consoles, Games & Accessories',
            'Nintendo DS Games, Consoles & Accessories', 'Wii Games, Consoles & Accessories',
            'Wii U Games, Consoles & Accessories', 'Nintendo 3DS & 2DS Consoles, Games & Accessories',
            'Mac Games & Accessories', 'PC Games & Accessories', 'Virtual Reality Hardware & Accessories',
            'Puzzles', 'Video Game Consoles & Accessories', 'Sony PSP Games, Consoles & Accessories'
        ]
    },
    {
        'hobby_category': 'Intellectual & Educational',
        'product_categories': [
            'Learning & Education Toys', 'Science Education Supplies', 'Puzzles', 'Lab & Scientific Products',
            'Industrial & Scientific'
        ]
    },
    {
        'hobby_category': 'DIY & Technical',
        'product_categories': [
            'Power Tools & Hand Tools', 'Tools & Home Improvement', 'Industrial Power & Hand Tools',
            'Measuring & Layout', 'Welding & Soldering', 'Hardware', 'Building Supplies',
            'Additive Manufacturing Products', 'Laptop Accessories', 'Computers',
            'Data Storage', 'Computer Networking', 'Office Electronics', 'Computers & Tablets',
            'Wearable Technology', 'Electrical Equipment', 'Laptop Bags', 'Paint, Wall Treatments & Supplies',
            'Tablet Replacement Parts', 'Tablet Accessories', 'Computer Monitors', 'Cell Phones & Accessories',
            'Legacy Systems', 'Computer External Components', 'Electronic Components'
        ]
    },
    {
        'hobby_category': 'Physical & Fitness',
        'product_categories': [
            'Sports & Fitness', 'Sports Nutrition Products', 'Wellness & Relaxation Products',
            'Health Care Products', 'Foot, Hand & Nail Care Products', 'Diet & Sports Nutrition',
            'Tricycles, Scooters & Wagons'
        ]
    },
    {
        'hobby_category': 'Outdoor & Nature',
        'product_categories': [
            'Outdoor Recreation', 'Travel Accessories', 'Suitcases',
            'Smart Home: Lawn and Garden', 'Lighting & Ceiling Fans', 'Horse Supplies',
            'Automotive Tools & Equipment'
        ]
    },
    {
        'hobby_category': 'Animals & Pets',
        'product_categories': [
            'Dog Supplies', 'Cat Supplies', 'Fish & Aquatic Pets',
            'Small Animal Supplies', 'Pet Bird Supplies', 'Reptiles & Amphibian Supplies',
            'Horse Supplies'
        ]
    },
    {
        'hobby_category': 'Content & Media',
        'product_categories': [
            'Camera & Photo', 'Video Projectors', 'Computers & Tablets',
            'Portable Audio & Video', 'Data Storage', 'Smart Home: Home Entertainment',
            'Office Electronics', 'Laptop Accessories', 'Laptop Bags'
        ]
    },
    {
        'hobby_category': 'Cleaning & Organizing',
        'product_categories': [
            'Vacuum Cleaners & Floor Care', 'Household Cleaning Supplies',
            'Janitorial & Sanitation Supplies', 'Home Storage & Organization',
            'Home Appliances', 'Ironing Products'
        ]
    },
    {
        'hobby_category': 'Entertainment & Leisure',
        'product_categories': [
            'Video Games', 'Toys & Games', 'Novelty Toys & Amusements', 'Finger Toys', "Kids' Play Trucks",
            'Televisions & Video Products', 'eBook Readers & Accessories', 'Toys & Games',
            "Kids' Play Trucks", 'Headphones & Earbuds', 'Portable Audio & Video', 'Building Toys',
            'Home Audio & Theater Products', 'Smart Home: Home Entertainment','Baby & Toddler Toys',
            'Puppets & Puppet Theaters', "Kids' Dress Up & Pretend Play", "Kids' Play Boats",
            "Kids' Electronics", "Kids' Play Cars & Race Cars", 'Sexual Wellness Products', "Kids' Play Trains & Trams",
            "Kids' Play Buses", 'Toy Vehicle Playsets', 'Slot Cars, Race Tracks & Accessories', "Kids' Play Tractors"
        ]
    },
    {
        'hobby_category': 'Social & Lifestyle',
        'product_categories': [
            'Party Supplies', 'Gift Wrapping Supplies', 'Party Decorations',
            'Seasonal Décor', 'Perfumes & Fragrances', 'Beauty & Personal Care',
            'Toys & Games', 'Novelty Toys & Amusements', 'Beauty Tools & Accessories',
            'Skin Care Products', 'Hair Care Products', 'Bath Products', 'Personal Care Products',
            'Oral Care Products', 'Vision Products', 'Home Décor Products',
            "Men's Shoes", "Women's Shoes", "Boys' Shoes", "Girls' Shoes", "Men's Watches",
            "Women's Watches", "Boys' Watches", "Girls' Watches", "Men's Accessories",
            "Women's Accessories", "Boys' Accessories", "Girls' Accessories", "Men's Clothing",
            "Women's Clothing", "Boys' Clothing", "Girls' Clothing", "Women's Handbags",
            "Women's Jewelry", "Boys' Jewelry", "Girls' Jewelry", 'Makeup'
        ]
    },
    {
        'hobby_category': 'Wellness & Spirituality',
        'product_categories': [
            'Wellness & Relaxation Products', 'Beauty Tools & Accessories', 'Skin Care Products',
            'Hair Care Products', 'Bath Products', 'Personal Care Products', 'Foot, Hand & Nail Care Products',
            'Oral Care Products', 'Vision Products', 'Beauty & Personal Care', 'Health Care Products',
            'Occupational Health & Safety Products', 'Health & Household'
        ]
    },
    {
        'hobby_category': 'Culinary & Beverage',
        'product_categories': [
            'Kitchen & Dining', 'Food Service Equipment & Supplies',
            'Kitchen & Bath Fixtures', 'Baby & Toddler Feeding Supplies'
        ]
    },
    {
        'hobby_category': 'Financial & Investment',
        'product_categories': [
            'Computers', 'Office Electronics', 'Data Storage', 'Computer Servers',
            'Security & Surveillance Equipment', 'Computer Networking', 'Laptop Accessories'
        ]
    },
    {
        'hobby_category': 'Collecting & Curating',
        'product_categories': [
            'Beading & Jewelry Making', 'Arts & Crafts Supplies', 'Toy Figures & Playsets',
            'Stuffed Animals & Plush Toys', 'Wall Art', 'Dolls & Accessories'
        ]
    },
    {
        'hobby_category': 'Travel & Exploration',
        'product_categories': [
            'Suitcases', 'Travel Accessories', 'Travel Duffel Bags', 'Travel Tote Bags',
            'Luggage Sets', 'GPS & Navigation', 'Outdoor Recreation', 'Luggage', 'Rain Umbrellas',
            'Power Transmission Products', 'Backpacks', 'Baby Travel Gear'
        ]
    },
    {
        'hobby_category': 'Automotive & Motorsports',
        'product_categories': [
            'Automotive Tools & Equipment', 'Automotive Performance Parts & Accessories',
            'Automotive Replacement Parts', 'Car Electronics & Accessories', 'Vehicle Electronics',
            'Oils & Fluids', 'Motorcycle & Powersports', 'Car Care', 'Automotive Exterior Accessories',
            'Automotive Interior Accessories', 'Heavy Duty & Commercial Vehicle Equipment',
            'Child Safety Car Seats & Accessories', 'RV Parts & Accessories', 'Automotive Paint & Paint Supplies'
        ]
    }
]


In [ ]:
for combo in positive_combos:
    entries = generate_hobby_category_specific_entries(_hobby_category=combo['hobby_category'],
                                 _product_categories=combo['product_categories'], score=0.75)

    hobbies_dataset.extend(entries)

### Generate negative samples by category

In [ ]:
negative_combos = []
for combo in positive_combos:
    negative_combos.append({
        'hobby_category': combo['hobby_category'],
        'product_categories': [category for category in list(products_by_category.keys())
                                if not category in combo['product_categories']]
    })

In [ ]:
for combo in negative_combos:
    entries = generate_hobby_category_specific_entries(n=400, _hobby_category=combo['hobby_category'],
                                 _product_categories=combo['product_categories'], score=0)
    hobbies_dataset.extend(entries)

In [ ]:
len(hobbies_dataset)

10487

In [ ]:
with open("tailored_to_hobbies_dataset.json", "w", encoding="utf-8") as f:
    json.dump(hobbies_dataset, f, indent=2, ensure_ascii=False)
print("✅ Dataset saved to tailored_to_hobbies_dataset.json with named fields.")

✅ Dataset saved to tailored_to_hobbies_dataset.json with named fields.


## **Occasion**

In [ ]:
occasions_dataset = []

### Generate good and bad samples

In [ ]:
def generate_occasion_generic_entries(n=100):
    entries = []
    for i in range(n):
        occasion = random.choice(occasions)
        prods = products_by_occasion[occasion]
        if prods != []:
            appropriate = sample(prods, 3)
            for product in appropriate:
                entries.append({
                        'occasion': occasion,
                        'product': product,
                        'suitability': 1
                    })
        prods = bad_products_by_occasion[occasion]
        if prods != []:
            inappropriate = sample(prods, 10)
            for product in inappropriate:
                entries.append({
                        'occasion': occasion,
                        'product': product,
                        'suitability': 0
                    })

    return entries

entries = generate_occasion_generic_entries(500)
occasions_dataset.extend(entries)


In [ ]:
len(occasions_dataset)

6037

### Generate samples for a specific occasion given product categories.

In [ ]:
def generate_occasion_specific_entries(_occasion, n=50, _product_categories=list(products_by_category.keys())):
    possible_products = []
    for cat in _product_categories:
        possible_products.extend(products_by_category[cat])

    dataset = []
    if possible_products != []:
        prods = sample(possible_products, n)
        for product in prods:
            dataset.append({
                "occasion": _occasion,
                "product": product,
                "suitability": 0.75
            })

    return dataset

### Generate positive samples by occasion

In [ ]:
positive_combos = [
    {
        'occasion': 'Baby Shower',
        'product_categories': ['Toilet Training Products', 'Nursery Furniture, Bedding & Décor']
    },
    {
        'occasion': 'New Baby',
        'product_categories': [cat for cat in list(products_by_category.keys())
                                if 'Baby' in cat]+['Toilet Training Products', 'Nursery Furniture, Bedding & Décor']
    },
    {
        'occasion': "Pregnancy",
        'product_categories': ['Pregnancy & Maternity Products']
    },
    {
        'occasion': "Housewarming",
        'product_categories': [cat for cat in list(products_by_category.keys())
                                                if 'Home' in cat or 'House' in cat]
    },
    {
        'occasion': "Halloween",
        'product_categories': ["Kids' Dress Up & Pretend Play"]
    }
]

In [ ]:
for combo in positive_combos:
    entries = generate_occasion_specific_entries(_occasion=combo['occasion'],
                                 _product_categories=combo['product_categories'])
    occasions_dataset.extend(entries)

In [ ]:
with open("tailored_to_occasions_dataset.json", "w", encoding="utf-8") as f:
    json.dump(occasions_dataset, f, indent=2, ensure_ascii=False)
print("✅ Dataset saved to tailored_to_occasions_dataset.json with named fields.")

✅ Dataset saved to tailored_to_occasions_dataset.json with named fields.


## **User Info & Relationship**

In [ ]:
user_relationship_dataset = []

### Generate samples for a specific user and relationship

In [ ]:
def generate_specific_samples(user_profiles):
    entries = []
    for user_profile in user_profiles:
        user = user_profile['user']
        relationship = user_profile['relationship']

        appropriate = []
        prods = products_by_age_and_gender[(get_age_group(user['age']), user['gender'])]
        if prods != []:
            appropriate.extend(sample(prods, 3))
        prods = products_by_relationship[relationship]
        if prods != []:
            appropriate.extend(sample(prods, 3))
        for product in appropriate:
            entries.append({
                        'age': user['age'],
                        'gender': user['gender'],
                        'relationship': relationship,
                        'product': product,
                        'suitability': 1
                    })
        inappropriate = []
        prods = bad_products_by_age_and_gender[(get_age_group(user['age']), user['gender'])]
        if prods != []:
            inappropriate.extend(sample(prods, 15))
        prods = bad_products_by_relationship[relationship]
        if prods != []:
            inappropriate.extend(sample(prods, 15))
        for product in inappropriate:
            entries.append({
                        'age': user['age'],
                        'gender': user['gender'],
                        'relationship': relationship,
                        'product': product,
                        'suitability': 0
                    })

    return entries

users = generate_user_profiles(n=200)
entries = generate_specific_samples(users)
user_relationship_dataset.extend(entries)


In [ ]:
len(user_relationship_dataset)

6636

In [ ]:
with open("tailored_to_user_relationship_dataset.json", "w", encoding="utf-8") as f:
    json.dump(user_relationship_dataset, f, indent=2, ensure_ascii=False)
print("✅ Dataset saved to tailored_to_user_relationship_dataset.json with named fields.")

✅ Dataset saved to tailored_to_user_relationship_dataset.json with named fields.


# **The Model**

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import numpy as np

In [9]:
def sample(population, k):
    return random.sample(population, min(k, len(population)))

In [10]:
def get_product_embed(product):
    product_text = f"Gift category: {product['category']}. Gift title: {product['title']}"
    return torch.tensor(sbert.encode([product_text])[0])

In [11]:
# Load Sentence-BERT
sbert = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Normalize age to [0, 1] (assuming 1-100)
def normalize_age(age):
    return (age - 1) / 100

# Encode gender to one-hot
def encode_gender(gender):
    if gender == "male":
        return [1, 0, 0]
    elif gender == "female":
        return [0, 1, 0]
    else:
        return [0, 0, 1]

def embed_hobbies(data):
    return torch.tensor(sbert.encode(data['hobbies']))

def embed_occasion(data):
    return torch.tensor(sbert.encode([data['occasion']])[0])

def embed_user_relationship(data):
    age, gender, relationship = data['age'], data['gender'], data['relationship']
    age_embed = torch.tensor([normalize_age(age)], dtype=torch.float32)
    gender_embed = torch.tensor(encode_gender(gender), dtype=torch.float32)

    relationship_embed = torch.tensor(sbert.encode([relationship])[0])

    return torch.cat([age_embed, gender_embed, relationship_embed])

# Convert a single sample to input tensor and label
def process_sample(data, embedder):
    product_embed = get_product_embed(data['product'])

    final_input = torch.cat([embedder(data), product_embed])
    label = torch.tensor([data['suitability']], dtype=torch.float32)

    return final_input, label

# Define model
class SuitabilityModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()  # output ∈ [0, 1]
        )

    def forward(self, x):
        return self.net(x)

# Training function
def train_model(model, X, y, epochs=10, lr=1e-3):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for xi, yi in zip(X, y):
            optimizer.zero_grad()
            pred = model(xi.unsqueeze(0))
            loss = loss_fn(pred, yi.unsqueeze(0))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss = {total_loss/len(X):.4f}")

with open('prods_embed_2.json', 'r') as f:
    products = json.load(f)

for product in products:
    product['embed'] = torch.tensor(product['embed'])

# Inference function (ranking products)
def rank_products(hobbies_model, occasions_model, user_relationship_model, data, products):
    with torch.no_grad():
        scores = []
        hobbies = embed_hobbies(data)
        occasion = embed_occasion(data)
        user_relationship = embed_user_relationship(data)

        for product in products:
            product_embed = product['embed']

            hobbies_score = 0
            for hobby in hobbies:
                input_vector = torch.cat([hobby, product_embed])
                hobbies_score = max(hobbies_model(input_vector.unsqueeze(0)).item(), hobbies_score)
            input_vector = torch.cat([occasion, product_embed])
            occasion_score = occasions_model(input_vector.unsqueeze(0)).item()

            input_vector = torch.cat([user_relationship, product_embed])
            user_relationship_score = user_relationship_model(input_vector.unsqueeze(0)).item()

            score = hobbies_score*0.5 + occasion_score*0.2 + user_relationship_score*0.3
            scores.append((score, product))

        return sorted(scores, key=lambda x: x[0], reverse=True)


In [ ]:
import torch.nn.functional as F

def get_user_embed(user_data, relationship, occasion):
    gender = ""
    if user_data['gender'] != 'other':
        gender = user_data['gender']
    intro = f"A gift for my {relationship.lower()}, a {user_data['age']}-year-old {user_data['gender']}"

    hobbies_text = ""
    if user_data['hobbies'] != []:
        hobbies_text = f" who enjoys {', '.join(user_data['hobbies'])}"

    user_text = intro + hobbies_text + f". It's for {occasion.lower()}."
    return torch.tensor(sbert.encode([user_text])[0])

def filter(user_data, relationship, occasion, products, n=1000):
    user_embed = get_user_embed(user_data, relationship, occasion)
    scores = []
    for product in products:
        score = F.cosine_similarity(user_embed, product['embed'], dim=0).item()
        scores.append((score, product))

    sorted_scores = sorted(scores, key=lambda x: x[0], reverse=True)[:n]
    return [tup[1] for tup in sorted_scores]

## **Hobbies Model**

In [ ]:
with open("tailored_to_hobbies_dataset.json", "r") as f:
    data_samples = json.load(f)

# Shuffle and split
random.shuffle(data_samples)
X, y = zip(*[process_sample(d, embed_hobbies) for d in data_samples])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train
input_dim = X[0].shape[0]
hobbies_model = SuitabilityModel(input_dim)
train_model(hobbies_model, X_train, y_train, epochs=8)

Epoch 1: Loss = 0.4632
Epoch 2: Loss = 0.3385
Epoch 3: Loss = 0.2628
Epoch 4: Loss = 0.2159
Epoch 5: Loss = 0.1911
Epoch 6: Loss = 0.1724
Epoch 7: Loss = 0.1651
Epoch 8: Loss = 0.1576


In [ ]:
checkpoint = torch.load('hobbies_model.pth')
hobbies_model = SuitabilityModel(input_dim=checkpoint['input_dim'])
hobbies_model.load_state_dict(checkpoint['model_state_dict'])
hobbies_model.eval()

In [ ]:
torch.save({
    'input_dim': 768,
    'model_state_dict': hobbies_model.state_dict()
}, 'hobbies_model.pth')

## **Occasions Model**

In [ ]:
with open("tailored_to_occasions_dataset.json", "r") as f:
    data_samples = json.load(f)

# Shuffle and split
random.shuffle(data_samples)
X, y = zip(*[process_sample(d, embed_occasion) for d in data_samples])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train
input_dim = X[0].shape[0]
occasions_model = SuitabilityModel(input_dim)
train_model(occasions_model, X_train, y_train)

Epoch 1: Loss = 0.3652
Epoch 2: Loss = 0.2370
Epoch 3: Loss = 0.1599
Epoch 4: Loss = 0.1141
Epoch 5: Loss = 0.0977
Epoch 6: Loss = 0.0775
Epoch 7: Loss = 0.0660
Epoch 8: Loss = 0.0581
Epoch 9: Loss = 0.0546
Epoch 10: Loss = 0.0480


In [13]:
checkpoint = torch.load('occasions_model.pth')
occasions_model = SuitabilityModel(input_dim=checkpoint['input_dim'])
occasions_model.load_state_dict(checkpoint['model_state_dict'])
occasions_model.eval()

SuitabilityModel(
  (net): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [ ]:
torch.save({
    'input_dim': 768,
    'model_state_dict': occasions_model.state_dict()
}, 'occasions_model.pth')

## **User Info & Relationship Model**

In [ ]:
with open("tailored_to_user_relationship_dataset.json", "r") as f:
    data_samples = json.load(f)

# Shuffle and split
random.shuffle(data_samples)
X, y = zip(*[process_sample(d, embed_user_relationship) for d in data_samples])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train
input_dim = X[0].shape[0]
input_dim
# user_relationship_model = SuitabilityModel(input_dim)
# train_model(user_relationship_model, X_train, y_train)

772

In [14]:
checkpoint = torch.load('user_relationship_model.pth')
user_relationship_model = SuitabilityModel(input_dim=checkpoint['input_dim'])
user_relationship_model.load_state_dict(checkpoint['model_state_dict'])
user_relationship_model.eval()

SuitabilityModel(
  (net): Sequential(
    (0): Linear(in_features=772, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [ ]:
torch.save({
    'input_dim': 772,
    'model_state_dict': user_relationship_model.state_dict()
}, 'user_relationship_model.pth')